# Tau `category2=1` Events

Load the Tau category CSV and list the events that ended up in `second_category/category1`.

In [1]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path(
    "/project/def-nahee/kbas/Graphnet-Applications/Metadata/CategoryInformation/"
    "String340MC/Full_Geometry_Tau_category2_events.csv"
)

df = pd.read_csv(CSV_PATH)

category1_events = (
    df.loc[df["category_value"].astype(str) == "1", ["RunID", "EventID"]]
    .sort_values(["RunID", "EventID"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
print(f"n_events = {len(category1_events)}")
display(category1_events)


n_events = 52


,RunID,EventID
0,4,13
1,4,15
2,4,34
3,4,50
4,4,52
5,4,54
6,4,61
7,4,65
8,4,70
9,4,73


In [2]:
unique_run_ids = sorted(category1_events["RunID"].unique().tolist())

print(f"n_unique_RunID = {len(unique_run_ids)}")
print(unique_run_ids)


n_unique_RunID = 2
[4, 5]


In [3]:
run_ids = [2, 8, 9, 11, 12, 208, 224, 326, 335, 707, 929]

tau_raw_base = Path(
    "/project/6008051/pone_simulation/"
    "MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator"
)

tau_file_paths = [tau_raw_base / f"gen_{run_id:03d}.i3.zst" for run_id in run_ids]

for path in tau_file_paths:
    print(path)


/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_002.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_008.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_009.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_011.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_012.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_208.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_224.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_326.i3.zst
/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_335

In [4]:
from icecube import dataio, LeptonInjector

event_properties_fields = [
    "totalEnergy",
    "zenith",
    "azimuth",
    "finalStateX",
    "finalStateY",
    "finalType1",
    "finalType2",
    "initialType",
    "x",
    "y",
    "z",
    "totalColumnDepth",
    "radius",
    "impactParameter",
]

rows = []

for tau_file_path in tau_file_paths:
    print(f"reading {tau_file_path}")
    i3_file = dataio.I3File(str(tau_file_path))
    n_daq = 0

    while i3_file.more():
        try:
            frame = i3_file.pop_daq()
        except Exception as exc:
            if "no frame to pop" in str(exc):
                break
            raise

        n_daq += 1

        if "I3EventHeader" not in frame or "EventProperties" not in frame:
            continue

        header = frame["I3EventHeader"]
        ep = frame["EventProperties"]

        row = {
            "raw_file": tau_file_path,
            "daq_index": n_daq,
            "RunID": header.run_id,
            "SubrunID": header.sub_run_id,
            "EventID": header.event_id,
            "SubEventID": header.sub_event_id,
        }
        for field in event_properties_fields:
            row[field] = getattr(ep, field)

        rows.append(row)

raw_event_properties_df = (
    pd.DataFrame(rows)
    .sort_values(["RunID", "EventID", "SubEventID"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)
print(f"n_rows = {len(raw_event_properties_df)}")
display(raw_event_properties_df)


reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_002.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_008.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_009.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_011.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_012.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_208.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_224.i3.zst
reading /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_326.i3.zst
reading /project/6008051/pone_simulation

,raw_file,daq_index,RunID,SubrunID,EventID,SubEventID,totalEnergy,zenith,azimuth,finalStateX,finalStateY,finalType1,finalType2,initialType,x,y,z,totalColumnDepth,radius,impactParameter
0,/project/6008051/pone_simulation/MC000004-nu_t...,1,2,4294967295,1,0,156.867477,1.657823,5.424200,0.101102,0.964635,-12,-2000001006,-12,1316.041866,-368.033636,-633.574018,4.315957e+05,1366.533919,924.625530
1,/project/6008051/pone_simulation/MC000004-nu_t...,2,2,4294967295,2,0,478.039248,2.114700,2.381347,0.040312,0.188911,-12,-2000001006,-12,15.154113,782.779503,-678.052788,5.215119e+05,782.926176,654.088491
2,/project/6008051/pone_simulation/MC000004-nu_t...,3,2,4294967295,3,0,4580.123618,1.731660,5.063093,0.301849,0.898826,-12,-2000001006,-12,1533.418714,-4306.264398,-693.100047,1.108505e+06,4571.136184,62.071172
3,/project/6008051/pone_simulation/MC000004-nu_t...,4,2,4294967295,4,0,208.495681,2.395227,5.944129,0.162586,0.634810,-12,-2000001006,-12,1120.614521,158.842985,-32.889327,3.248163e+05,1131.816239,885.379849
4,/project/6008051/pone_simulation/MC000004-nu_t...,5,2,4294967295,5,0,811.895147,1.793470,3.239037,0.273680,0.241067,-12,-2000001006,-12,155.248640,-429.581706,24.598157,5.413720e+05,456.774104,442.648893
5,/project/6008051/pone_simulation/MC000004-nu_t...,6,2,4294967295,6,0,3807.984143,2.371174,4.696336,0.344598,0.076246,-12,-2000001006,-12,-342.622785,-2007.340277,-1644.818226,1.040683e+06,2036.370634,430.785784
6,/project/6008051/pone_simulation/MC000004-nu_t...,7,2,4294967295,7,0,521.603327,1.709602,3.448966,0.101779,0.070637,-12,-2000001006,-12,-2022.488089,142.537853,-287.437653,4.492610e+05,2027.504651,748.155717
7,/project/6008051/pone_simulation/MC000004-nu_t...,8,2,4294967295,8,0,201265.089275,1.964729,3.152035,0.016386,0.175555,-12,-2000001006,-12,-6667.191358,-610.611320,-2452.317972,2.838150e+06,6695.094233,617.051326
8,/project/6008051/pone_simulation/MC000004-nu_t...,9,2,4294967295,9,0,184.088782,0.851253,2.067088,0.110334,0.150809,-12,-2000001006,-12,-329.354483,231.582842,1307.941677,3.136058e+05,402.622638,767.398130
9,/project/6008051/pone_simulation/MC000004-nu_t...,10,2,4294967295,10,0,169.991681,0.805374,0.909918,0.094826,0.662019,-12,-2000001006,-12,486.803079,-630.781084,-791.476899,4.303830e+05,796.782288,884.564300


In [5]:
print(f"total DAQ/EventProperties rows = {len(raw_event_properties_df)}")
print(f"unique RunID count = {raw_event_properties_df['RunID'].nunique()}")
print(f"unique EventID count = {raw_event_properties_df[['RunID', 'EventID', 'SubEventID']].drop_duplicates().shape[0]}")

for column in ["finalType1", "finalType2", "initialType"]:
    values = raw_event_properties_df[column].drop_duplicates().tolist()
    print(f"\nunique {column} ({len(values)}):")
    print(values)


total DAQ/EventProperties rows = 2200
unique RunID count = 11
unique EventID count = 2200

unique finalType1 (2):
[-12, 12]

unique finalType2 (1):
[-2000001006]

unique initialType (2):
[-12, 12]


to move:

mv /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_002.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_008.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_009.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_011.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_012.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_208.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_224.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_326.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_335.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_707.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Tau_PMT_Response/tau_gen_929.i3.gz /home/kbas/scratch/String340MC/Full_Geometry/Electron_PMT_Response/

to move

mv /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_002.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_008.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_009.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_011.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_012.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_208.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_224.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_326.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_335.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_707.i3.gz /home/kbas/scratch/String340MC/102_string/Tau_PMT_Response/tau_tau_gen_929.i3.gz /home/kbas/scratch/String340MC/102_string/Electron_PMT_Response/